> 在线讲义：[第 13 章 · 多头注意力 multi-head attention](https://wikieden.github.io/transformer-visual/lessons/13-multi-head.html) · 本书《从零读懂 Transformer》

# 第 13 章 · 多头注意力 multi-head attention

配套讲义：[在线版](https://wikieden.github.io/transformer-visual/lessons/13-multi-head.html) · 仓库里 `transformer-deep-dive/notes/lessons/13-multi-head.md`

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 torch 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。不需要 GPU。

In [ ]:
import torch

torch.set_printoptions(precision=3, sci_mode=False)  # 打印保留 3 位小数，不用科学计数法
print("torch", torch.__version__)

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一行代码、跑出来是什么。

| 讲义 | 公式 / 说法 | PyTorch 写法 | 跑出来 |
|---|---|---|---|
| 一个头会混 | 小红 0.5、因为 0.5 | `[0.5, 0.5] @ ...` | [5, 5] |
| 拼接不是相加 | Concat | `torch.cat([h1, h2])` | [10, 0, 0, 10]；相加是 [10, 10] |
| 切成 8 个头 | 每头 512 / 8 个数 | `x.view(n, 8, 64).transpose(0, 1)` | (8, 3, 64) |
| 拼回去 | Concat | `.transpose(0, 1).reshape(n, 512)` | 和原来一模一样 |
| 多头整层 | $\text{Concat}(\text{head}_1,\dots)W^O$ | 切头 → SDPA → 拼接 → `@ w_o` | (3, 512) |
| 参数和头数无关 | $4 d_{model}^2$ | `nn.MultiheadAttention(8, h, bias=False)` | h = 1、2、4、8 都是 256 = 4 × 8² |

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
import torch
import torch.nn.functional as F
from torch import nn

**函数 `split_heads`**：n × d → h × n × (d/h)：每个头拿一段。

In [ ]:
def split_heads(x: torch.Tensor, h: int) -> torch.Tensor:
    """n × d → h × n × (d/h)：每个头拿一段。"""
    n, d = x.shape
    return x.view(n, h, d // h).transpose(0, 1)

**函数 `merge_heads`**：h × n × d_k → n × (h·d_k)：首尾拼接（Concat）。

In [ ]:
def merge_heads(x: torch.Tensor) -> torch.Tensor:
    """h × n × d_k → n × (h·d_k)：首尾拼接（Concat）。"""
    h, n, d_k = x.shape
    return x.transpose(0, 1).reshape(n, h * d_k)

**函数 `multi_head`**：小工具，把结果整理成好读的样子再打印。

In [ ]:
def multi_head(x: torch.Tensor, w_q, w_k, w_v, w_o, h: int) -> torch.Tensor:
    q, k, v = (split_heads(x @ w, h) for w in (w_q, w_k, w_v))
    heads = F.scaled_dot_product_attention(q, k, v)  # 每个头各自 softmax
    return merge_heads(heads) @ w_o

**函数 `param_count`**：小工具，把结果整理成好读的样子再打印。

In [ ]:
def param_count(d: int, h: int) -> int:
    m = nn.MultiheadAttention(d, h, bias=False)
    return sum(p.numel() for p in m.parameters())

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
one = torch.tensor([0.5, 0.5]) @ torch.tensor([[10.0, 0.0], [0.0, 10.0]])
print("一个头：小红 0.5、因为 0.5 →", one.tolist(), "（混在一起）")
head1, head2 = torch.tensor([10.0, 0.0]), torch.tensor([0.0, 10.0])
print("两个头拼接 torch.cat:", torch.cat([head1, head2]).tolist(), "  相加是", (head1 + head2).tolist())

In [ ]:
print("\n论文尺寸：512 个数切成 8 个头")
torch.manual_seed(0)
x = torch.randn(3, 512)
print("  split_heads 形状:", tuple(split_heads(x, 8).shape), "（8 个头 × 3 个词 × 每头 512 / 8 个数）")
print("  拼回去一样吗:", torch.equal(merge_heads(split_heads(x, 8)), x))
ws = [torch.randn(512, 512) / 512**0.5 for _ in range(4)]
print("  多头输出形状:", tuple(multi_head(x, *ws, h=8).shape))

In [ ]:
print("\n参数量和头数无关（小例子 d = 8，不算 bias）:")
for h in [1, 2, 4, 8]:
    print(f"  h = {h}: {param_count(8, h)} 个参数")

## 3. 代码里才看得到的

1. **多头只是"换个形状看"**：`view` 把每个词的 512 个数看成 8 段 × 64，不复制任何数；8 个头一起交给 attention，每个头的 softmax 自动各算各的。
2. **`view` 之后 `transpose` 再拼回去要用 `reshape`**：转置后数在内存里不再按顺序排，`view` 会报错，`reshape` 会在需要时自己复制一份。
3. **`nn.MultiheadAttention` 的参数**：$W^Q$、$W^K$、$W^V$ 竖着摞在一个 `in_proj_weight`（3d × d）里，$W^O$ 在 `out_proj` 里，合起来正好 $4d^2$，头数只决定怎么切。

## 4. 自己改一改

- 把上面任意一格里的数字改一改，再按 Shift + Enter，看结果怎么变。

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 一头只有一组比，多头各看各的戏；拼回原长再汇总。